# Tugas Praktikum - SVM

Notebook ini berisi dua bagian tugas:

1. **Klasifikasi gender pada data `voice.csv`** menggunakan SVM dengan kernel **linear**, **polynomial**, dan **RBF**,
   masing-masing pada rasio split **70:30** dan **80:20**.
2. **Klasifikasi citra siang dan malam** (data Praktikum 5) menggunakan SVM kernel **RBF** dengan fitur **histogram**,
   rasio **80:20**, disertai **hyperparameter tuning**.

## Bagian 1 - Klasifikasi Data Voice

### Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

%matplotlib inline
RANDOM_STATE = 42

### Load Dataset

In [2]:
df = pd.read_csv('voice.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (3168, 21)


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [3]:
# Informasi dataset
print('Jumlah data duplikat :', df.duplicated().sum())
print('Jumlah nilai kosong  :', df.isnull().sum().sum())
print('\nDistribusi label:')
print(df['label'].value_counts())

Jumlah data duplikat : 2
Jumlah nilai kosong  : 0

Distribusi label:
label
male      1584
female    1584
Name: count, dtype: int64


### Pra Pengolahan Data

In [4]:
# Hapus data duplikat
df = df.drop_duplicates().reset_index(drop=True)

# Pisahkan fitur dan label, lalu encode label (female=0, male=1)
X = df.drop(columns='label')
y = LabelEncoder().fit_transform(df['label'])

print(f'Shape setelah pembersihan: {X.shape}')
print(f'Distribusi label: {np.bincount(y)}')

Shape setelah pembersihan: (3166, 20)
Distribusi label: [1583 1583]


### Pemodelan SVM

Setiap model menggunakan `StandardScaler` + `SVC` (pipeline), karena SVM sensitif terhadap skala fitur.
Kernel yang diuji: `linear`, `poly` (derajat 3), dan `rbf`.

In [5]:
results = []

splits = {
    '70:30': 0.30,
    '80:20': 0.20,
}

kernels = ['linear', 'poly', 'rbf']

for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    for kernel in kernels:
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel=kernel))
        ])
        model.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))

        results.append({
            'Split': split_name,
            'Kernel': kernel,
            'Train Accuracy': train_acc,
            'Test Accuracy': test_acc,
        })

results_df = pd.DataFrame(results)
results_df

,Split,Kernel,Train Accuracy,Test Accuracy
0,70:30,linear,0.976083,0.973684
1,70:30,poly,0.966155,0.957895
2,70:30,rbf,0.985560,0.981053
3,80:20,linear,0.975908,0.973186
4,80:20,poly,0.966825,0.962145
5,80:20,rbf,0.984992,0.982650


### Tabel Performansi (Akurasi)

In [6]:
# Tabel akurasi test berdasarkan split dan kernel
pivot_acc = results_df.pivot(index='Kernel', columns='Split', values='Test Accuracy')
pivot_acc = pivot_acc[['70:30', '80:20']]
pivot_acc

Split,70:30,80:20
Kernel,,
linear,0.973684,0.973186
poly,0.957895,0.962145
rbf,0.981053,0.982650


In [7]:
# Tabel lengkap (train & test)
results_df.sort_values(['Split', 'Kernel']).reset_index(drop=True).style.format({
    'Train Accuracy': '{:.4f}',
    'Test Accuracy': '{:.4f}',
})

,Split,Kernel,Train Accuracy,Test Accuracy
0,70:30,linear,0.9761,0.9737
1,70:30,poly,0.9662,0.9579
2,70:30,rbf,0.9856,0.9811
3,80:20,linear,0.9759,0.9732
4,80:20,poly,0.9668,0.9621
5,80:20,rbf,0.9850,0.9826


**Catatan hasil Bagian 1:** performansi ketiga kernel sangat tinggi dan berdekatan. Kernel linear dan RBF
umumnya memberikan akurasi terbaik pada dataset voice, sedangkan kernel polynomial cenderung sedikit lebih rendah.
Peningkatan rasio data latih (80:20) umumnya menghasilkan akurasi yang sedikit lebih stabil.

## Bagian 2 - Klasifikasi Siang/Malam dengan Fitur Histogram dan SVM RBF

Data yang digunakan adalah data citra Praktikum 5 (`images/`) yang telah diekstraksi.
Fitur yang digunakan adalah **histogram warna pada ruang HSV** (32 bin per kanal H, S, dan V),
sehingga setiap citra direpresentasikan oleh 96 fitur. Histogram dinormalisasi agar jumlah tiap kanal = 1.

### Load Data dan Ekstraksi Fitur Histogram

In [8]:
from pathlib import Path
import matplotlib.image as mpimg
import cv2

all_dir = 'images/'

def load_paths(img_dir):
    p = Path(img_dir)
    items = []

    for file in sorted(p.rglob('*.jpg')):
        label = file.parent.name
        if label in ('day', 'night'):
            items.append((str(file), label))

    return items

# Gabungkan seluruh data (training + test) agar dapat dibagi ulang 80:20
image_items = load_paths(all_dir)
print(f'Total citra: {len(image_items)}')
print('Contoh:', image_items[0])

Total citra: 400
Contoh: ('images/test/day/20151101_152050.jpg', 'day')


In [9]:
def histogram_feature(image, bins=32):
    # Pastikan citra dalam format RGB
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    # Standarisasi ukuran seperti Praktikum 5, lalu konversi ke HSV
    image = cv2.resize(image, (1100, 600))
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    features = []
    for channel, value_range in enumerate([[0, 180], [0, 256], [0, 256]]):
        hist = cv2.calcHist([hsv], [channel], None, [bins], value_range).flatten()
        hist = hist / (hist.sum() + 1e-8)  # normalisasi
        features.append(hist)

    return np.concatenate(features)

In [10]:
X_img = []
y_img = []

for path, label in image_items:
    img = mpimg.imread(path)
    if img is None:
        continue
    X_img.append(histogram_feature(img))
    y_img.append(1 if label == 'day' else 0)

X_img = np.array(X_img)
y_img = np.array(y_img)

print(f'Shape fitur : {X_img.shape}')
print(f'Distribusi  : day={np.sum(y_img == 1)}, night={np.sum(y_img == 0)}')

Shape fitur : (400, 96)
Distribusi  : day=200, night=200


### Split Data 80:20

In [11]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img, test_size=0.20, random_state=RANDOM_STATE, stratify=y_img
)

print(f'Data latih : {X_train_img.shape[0]} citra')
print(f'Data uji   : {X_test_img.shape[0]} citra')

Data latih : 320 citra
Data uji   : 80 citra


### Model SVM RBF Baseline

Sebagai pembanding, model RBF dengan hyperparameter default (`C=1`, `gamma='scale'`) dilatih terlebih dahulu.

In [12]:
baseline_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])
baseline_rbf.fit(X_train_img, y_train_img)

baseline_train_acc = accuracy_score(y_train_img, baseline_rbf.predict(X_train_img))
baseline_test_acc = accuracy_score(y_test_img, baseline_rbf.predict(X_test_img))

print(f'Baseline RBF - Train Accuracy: {baseline_train_acc:.4f}')
print(f'Baseline RBF - Test Accuracy : {baseline_test_acc:.4f}')

Baseline RBF - Train Accuracy: 1.0000
Baseline RBF - Test Accuracy : 1.0000


### Hyperparameter Tuning Kernel RBF

In [13]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100, 1000],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1, 1],
    'svm__class_weight': [None, 'balanced'],
}

grid_search = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf'))
    ]),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_img, y_train_img)

print('Best parameters :', grid_search.best_params_)
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits


Best parameters : {'svm__C': 100, 'svm__class_weight': None, 'svm__gamma': 0.001}
Best CV accuracy: 0.9969


In [14]:
best_rbf = grid_search.best_estimator_

tuned_train_acc = accuracy_score(y_train_img, best_rbf.predict(X_train_img))
tuned_test_acc = accuracy_score(y_test_img, best_rbf.predict(X_test_img))

print(f'Tuned RBF - Train Accuracy: {tuned_train_acc:.4f}')
print(f'Tuned RBF - Test Accuracy : {tuned_test_acc:.4f}')

y_pred_tuned = best_rbf.predict(X_test_img)
print('\nClassification Report (data uji):')
print(classification_report(y_test_img, y_pred_tuned, target_names=['night', 'day']))
print('Confusion Matrix:')
print(confusion_matrix(y_test_img, y_pred_tuned))

Tuned RBF - Train Accuracy: 1.0000
Tuned RBF - Test Accuracy : 1.0000

Classification Report (data uji):
              precision    recall  f1-score   support

       night       1.00      1.00      1.00        40
         day       1.00      1.00      1.00        40

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

Confusion Matrix:
[[40  0]
 [ 0 40]]


### Tabel Perbandingan Performansi

In [15]:
hist_summary = pd.DataFrame({
    'Model': ['SVM RBF Baseline', 'SVM RBF Tuned'],
    'Hyperparameter': ['C=1, gamma=scale', str(grid_search.best_params_)],
    'Train Accuracy': [baseline_train_acc, tuned_train_acc],
    'Test Accuracy': [baseline_test_acc, tuned_test_acc],
})

hist_summary.style.format({
    'Train Accuracy': '{:.4f}',
    'Test Accuracy': '{:.4f}',
})

,Model,Hyperparameter,Train Accuracy,Test Accuracy
0,SVM RBF Baseline,"C=1, gamma=scale",1.0000,1.0000
1,SVM RBF Tuned,"{'svm__C': 100, 'svm__class_weight': None, 'svm__gamma': 0.001}",1.0000,1.0000


**Catatan hasil Bagian 2:** fitur histogram warna HSV mampu memisahkan citra siang dan malam dengan baik.
Hyperparameter tuning (`C`, `gamma`, `class_weight`) dapat meningkatkan akurasi dibandingkan model baseline.
Nilai `gamma` yang terlalu besar berpotensi overfitting (akurasi latih tinggi, akurasi uji turun), sehingga
pemilihan hyperparameter melalui cross-validation penting dilakukan.

## Kesimpulan

In [16]:
print('=== Bagian 1: voice.csv ===')
for _, row in results_df.iterrows():
    print(f"Split {row['Split']:<5} | Kernel {row['Kernel']:<6} | "
          f"Train {row['Train Accuracy']:.4f} | Test {row['Test Accuracy']:.4f}")

print('\n=== Bagian 2: Citra Siang/Malam (Histogram + RBF) ===')
print(f'Baseline RBF | Train {baseline_train_acc:.4f} | Test {baseline_test_acc:.4f}')
print(f'Tuned RBF    | Train {tuned_train_acc:.4f} | Test {tuned_test_acc:.4f}')
print('Best params  :', grid_search.best_params_)

=== Bagian 1: voice.csv ===
Split 70:30 | Kernel linear | Train 0.9761 | Test 0.9737
Split 70:30 | Kernel poly   | Train 0.9662 | Test 0.9579
Split 70:30 | Kernel rbf    | Train 0.9856 | Test 0.9811
Split 80:20 | Kernel linear | Train 0.9759 | Test 0.9732
Split 80:20 | Kernel poly   | Train 0.9668 | Test 0.9621
Split 80:20 | Kernel rbf    | Train 0.9850 | Test 0.9826

=== Bagian 2: Citra Siang/Malam (Histogram + RBF) ===
Baseline RBF | Train 1.0000 | Test 1.0000
Tuned RBF    | Train 1.0000 | Test 1.0000
Best params  : {'svm__C': 100, 'svm__class_weight': None, 'svm__gamma': 0.001}
